# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nadim7009/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Lane 4, CTR opportunity scoring.** The contract in five plain answers:

1. **One row means:** one page (`client_hash_id` + `content_hash_id`), summarised over a short window of daily Search Console data.
2. **Table:** `fact_content_daily_performance`, only the `month=2026-03` partition for now. That's a mid-panel month, so the final month (June 2026) stays sealed as a test month.
3. **Time window:** features come from **1 to 15 March 2026** (the decision moment is the end of 15 March). The outcome comes from **16 to 31 March 2026**. The two windows don't overlap.
4. **What I predict:** each page's CTR in the outcome window (clicks ÷ impressions, 16 to 31 March), an observed future outcome. The ranking later compares a page's actual CTR with this expected CTR.
5. **Deliberately excluded:** anything measured in 16 to 31 March as a feature, and `fact_content_query_90d`, whose fixed 90-day window ends at the panel's end. It would mix later months into March features.

The setup cell connects to the release without ever printing the token. The next cell reads the partition's schema, so the flag names used below come from the data itself.

In [ ]:
%pip -q install duckdb
import os, duckdb, pandas as pd, numpy as np
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

# Token: env var, then Colab Secret named HF_TOKEN. Never typed into a cell (the repo is public).
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Add a Colab Secret named HF_TOKEN (key icon on the left), then re-run."

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
del HF_TOKEN

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
print("Connected. Working on one partition: month=2026-03")

In [ ]:
# Read the schema of the March partition (metadata only), and find the availability flags by name
schema = con.sql(f"DESCRIBE SELECT * FROM {MARCH}").df()[["column_name", "column_type"]]
print(schema.to_string(index=False))

flags = [c for c in schema["column_name"] if c.endswith("_available")]
GSC_FLAG = next(c for c in flags if c.startswith("gsc"))
print(f"\nAvailability flags found: {flags}  -> using {GSC_FLAG} for Search Console rows")

## 2. Fields: feature / label / context / excluded

| Field | Bucket | Why |
| --- | --- | --- |
| `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` for 1 to 15 March | Feature (after aggregating) | Measured before the decision moment |
| `gsc_impressions`, `gsc_clicks` for 16 to 31 March | Label | They make up the outcome CTR I predict |
| `report_date` | Context | Places each row in the feature or outcome window; never a feature value |
| `client_hash_id`, `content_hash_id` | Context | Grain, joins and client-grouped splits only; pseudonyms carry no meaning |
| GSC availability flag | Context (filter) | Rows count only when it `IS TRUE`. `= FALSE` or `NOT` would mishandle NULL flags |
| GA4 columns (sessions, engagement) | Excluded for now | My lane is about clicks from search. GA4 is zero-filled before each client's GA4 start, which needs its own filter |
| `fact_content_query_90d` | Excluded for now | Its fixed 90-day window ends with the panel, not in March, so it would leak later months into March features |
| Any 16 to 31 March metric used as a feature | Excluded | That's the outcome; section 3 shows what happens if one sneaks in |

## 3. Verify it with queries (grain, counts, missing values, windows)

Three queries, each on the March partition only.

**Query 1, grain.** The contract says one daily row = one date × client × page. If that holds, no combination appears twice.

In [ ]:
dupes = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {MARCH}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f"Duplicate (date, client, page) combinations found: {len(dupes)}")
print("Grain holds: one row = one page on one day" if dupes.empty else dupes)

**Query 2, row count and date span of the slice.**

In [ ]:
span = con.sql(f"""
    SELECT COUNT(*)                         AS daily_rows,
           MIN(report_date)                 AS first_day,
           MAX(report_date)                 AS last_day,
           COUNT(DISTINCT report_date)      AS days,
           COUNT(DISTINCT client_hash_id)   AS clients,
           COUNT(DISTINCT content_hash_id)  AS pages
    FROM {MARCH}
""").df()
print(span.T.to_string(header=False))

**Query 3, availability.** How many March rows actually carry Search Console data? The flag is filtered with `IS TRUE`, because the docs warn it can be TRUE, FALSE or NULL.

In [ ]:
avail = con.sql(f"""
    SELECT COUNT(*)                                              AS all_rows,
           COUNT(*) FILTER (WHERE {GSC_FLAG} IS TRUE)            AS gsc_available_true,
           COUNT(*) FILTER (WHERE {GSC_FLAG} IS FALSE)           AS gsc_available_false,
           COUNT(*) FILTER (WHERE {GSC_FLAG} IS NULL)            AS gsc_available_null,
           COUNT(*) FILTER (WHERE {GSC_FLAG} IS TRUE AND gsc_impressions > 0) AS true_with_impressions
    FROM {MARCH}
""").df()
print(avail.T.to_string(header=False))
print(f"\nRows that survive the IS TRUE filter: {avail['gsc_available_true'][0] / avail['all_rows'][0]:.1%}")

### Five features, and when each one is knowable

The feature frame is built in SQL, so only the small per-page result comes back. Each page needs at least 100 impressions in both windows (below that one click swings CTR too much) and an average position of 20 or better, matching my lane.

| Feature (1 to 15 March) | Knowable at the decision moment (end of 15 March) because… |
| --- | --- |
| `imp_f`: total impressions | Search Console reports it daily, and all of these days are on or before 15 March |
| `ctr_f`: clicks ÷ impressions × 100 | Built only from 1 to 15 March clicks and impressions |
| `pos_f`: mean daily average position | The daily position for each day up to 15 March is already recorded |
| `pos_sd_f`: day-to-day spread of position | Uses the same 1 to 15 March position values; a page bouncing between ranks gets a less stable CTR |
| `days_f`: days with any impressions | A count of days up to 15 March, so it's fixed by the decision moment |

The target, `ctr_next` (16 to 31 March CTR), is built in the same query but used only as the label.

In [ ]:
frame = con.sql(f"""
    WITH f AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions)                              AS imp_f,
               SUM(gsc_clicks)                                   AS clk_f,
               AVG(gsc_avg_position)                             AS pos_f,
               STDDEV_SAMP(gsc_avg_position)                     AS pos_sd_f,
               COUNT(*) FILTER (WHERE gsc_impressions > 0)       AS days_f
        FROM {MARCH}
        WHERE {GSC_FLAG} IS TRUE AND report_date <= DATE '2026-03-15'
        GROUP BY 1, 2
    ),
    o AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp_next,
               SUM(gsc_clicks)      AS clk_next
        FROM {MARCH}
        WHERE {GSC_FLAG} IS TRUE AND report_date >= DATE '2026-03-16'
        GROUP BY 1, 2
    )
    SELECT f.*, o.imp_next, o.clk_next
    FROM f JOIN o USING (client_hash_id, content_hash_id)
    WHERE f.imp_f >= 100 AND o.imp_next >= 100 AND f.pos_f > 0 AND f.pos_f <= 20
""").df()

frame["ctr_f"] = 100 * frame["clk_f"] / frame["imp_f"]
frame["ctr_next"] = 100 * frame["clk_next"] / frame["imp_next"]       # label, never a feature
frame["pos_sd_f"] = frame["pos_sd_f"].fillna(0)

FEATURES = ["imp_f", "ctr_f", "pos_f", "pos_sd_f", "days_f"]
assert not frame.duplicated(["client_hash_id", "content_hash_id"]).any(), "grain broken: duplicate pages"
print(f"Feature frame: {len(frame):,} pages from {frame['client_hash_id'].nunique()} clients")
frame[FEATURES + ["ctr_next"]].describe().round(2)

### The trap, sprung on purpose

The quick score is a gradient-boosting model predicting `ctr_next` from the five features. It's tested on clients it never saw (5 client-grouped folds, seed 42), and scored as R², the share of the outcome's spread explained.

Then I add **one label-derived column on purpose**: `clk_next`, the clicks in the outcome window. CTR next is just `clk_next` ÷ `imp_next`, so this column is the answer in disguise.

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score

def quick_score(cols):
    X, y, g = frame[cols], frame["ctr_next"], frame["client_hash_id"]
    scores = []
    for tr, te in GroupKFold(n_splits=5).split(X, y, g):
        m = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, random_state=42)
        m.fit(X.iloc[tr], y.iloc[tr])
        scores.append(r2_score(y.iloc[te], m.predict(X.iloc[te])))
    return np.mean(scores), scores

honest, honest_folds = quick_score(FEATURES)
leaky, leaky_folds = quick_score(FEATURES + ["clk_next"])    # the deliberate leak

print(f"Honest R² (5 features):             {honest:.3f}   folds {np.round(honest_folds, 3)}")
print(f"Leaky R² (+ clk_next, from label):   {leaky:.3f}   folds {np.round(leaky_folds, 3)}")

The leaky score jumps toward perfect. That isn't skill: `clk_next` only exists after 31 March, so on 15 March, when the editor would use the ranking, it can't be known. It's the same lesson as notebook 02's `trend_pct`, this time on warehouse data.

**The leak is deleted. The number I keep is the honest R² from the five features.** The cell below removes the column from the frame so nothing later can pick it up by accident.

In [ ]:
frame = frame.drop(columns=["clk_next"])
assert "clk_next" not in frame.columns
print(f"Leak removed. Honest number kept: R² = {honest:.3f} on held-out clients, 5 features.")

## 4. Data limits

**Named limitation: two half-months are a short window.** The features cover 15 days and the outcome 16 days, both inside one month. A page's CTR over two weeks moves with things this slice can't see: the queries it happened to rank for, and how search results looked those days. Weekday mix, seasonality and pages that only got going late in the month are other examples. So this frame can show whether CTR is predictable from one half-month to the next. It can't show that a gap lasts for months.

Other limits I'm carrying forward:

- **Unbalanced panel.** Clients start their history on different dates, so some pages are missing from March because tracking hadn't started, not because they got no traffic. The check below shows how concentrated the frame is.
- **Average of daily positions.** It treats a day with 5 impressions the same as a day with 5,000.
- **No query mix yet.** The query table can't be used for March without leaking later months (section 2).

In [ ]:
by_client = frame["client_hash_id"].value_counts()
print(f"Pages per client in the frame: largest client holds {by_client.iloc[0] / len(frame):.0%}; "
      f"{(by_client < 20).sum()} of {len(by_client)} clients have fewer than 20 pages")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.